In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
!sh /content/gdrive/MyDrive/YOUR_FOLDER/install_script

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")
!sh /content/gdrive/MyDrive/YOUR_FOLDER/install_script
import os
from flask import Flask, render_template, request
from markupsafe import escape
import cje1gw
import sqlite3

os.chdir("/content/gdrive/MyDrive/YOUR_FOLDER/")

app = Flask(__name__)
cje1gw.run_with(app)

@app.route("/report2.html")
def e47():
    return render_template("report2.html")

@app.route("/search", methods=["POST"])
def search():
    keyword = request.form["word"]
    field = request.form["field"]
    keywords = keyword.strip().split()

    con = sqlite3.connect("/content/gdrive/MyDrive/YOUR_FOLDER/opaclarge.db")
    cur = con.cursor()

    where_clauses = []
    params = []

    for word in keywords:
        if field == "any":
            where_clauses.append("(title LIKE ? OR creator LIKE ?)")
            params.extend(['%' + word + '%', '%' + word + '%'])
        elif field == "title":
            where_clauses.append("title LIKE ?")
            params.append('%' + word + '%')
        elif field == "creator":
            where_clauses.append("creator LIKE ?")
            params.append('%' + word + '%')

    sql = "SELECT * FROM opaclarge WHERE " + " AND ".join(where_clauses)

    s = """<!DOCTYPE html><html><head><meta charset="UTF-8"><title>検索結果</title>
    <style>
        h2 {color: navy;}
        p {color: #333;}
        .label {color: darkgreen; font-weight: bold;}
    </style></head><body>"""

    s += f"<h2>「{escape(keyword)}」の検索結果</h2>\n"

    for row in cur.execute(sql, params):
        s += "<p>"
        s += f"<span class='label'> NDL識別子:</span> {escape(row[0])}　"
        s += f"<span class='label'> タイトル:</span> {escape(row[1])}　"
        s += f"<span class='label'> 著者・作者:</span> {escape(row[2])}"
        s += f"<span class='label'> 出版社:</span> {escape(row[3])}　"
        s += f"<span class='label'> 出版年月:</span> {escape(row[4])}　"
        s += f"<span class='label'> ISBN:</span> {escape(row[5])}"
        s += "</p>\n"

    s += "</body></html>"
    con.close()
    return s

if __name__ == "__main__":
    app.run()